# Spike C — execution-graph (Ch7 Self-Evolution)

Foundation of Ch7 self-awareness. We trace a DevOps investigation agent through one query — multiple LLM calls, retrievals, tool invocations — including one tool call that fails. The execution graph preserves the structure of the failed call AND its causal chain to root.

**Source:** *Agentic Graph RAG* Ch7 — The Foundation of Self-Awareness + Example 7-1.

In [ ]:
import os, sys, json
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SKILL_DIR = os.path.join(REPO_ROOT, "skills", "self-evolution", "execution-graph")
sys.path.insert(0, SKILL_DIR)

from lib import ExecutionGraph

os.environ.update({
    "AWS_ACCESS_KEY_ID": "testing", "AWS_SECRET_ACCESS_KEY": "testing",
    "AWS_SECURITY_TOKEN": "testing", "AWS_SESSION_TOKEN": "testing",
    "AWS_DEFAULT_REGION": "us-east-1",
})
from moto import mock_aws
import boto3
from botocore.exceptions import ClientError
FICTIONAL_ACCOUNT = "123456789012"
print("setup ok")

## Trace the agent's investigation against moto-mocked AWS

In [ ]:
g = ExecutionGraph(execution_id="investigate-checkout-latency-2026-03-15")

with mock_aws():
    ec2 = boto3.client("ec2", region_name="us-east-1")
    cw = boto3.client("cloudwatch", region_name="us-east-1")
    cw.put_metric_data(
        Namespace="AWS/ApplicationELB",
        MetricData=[{"MetricName": "TargetResponseTime", "Value": 8.2, "Unit": "Seconds"}],
    )

    # ROOT
    root = g.begin_node("Decision_Point", input_payload={"query": "Why is checkout latency spiking?"})
    g.complete_node(root, output_payload={"plan": "check metrics, check deploys, hypothesize"},
                    latency_ms=12.3, token_count=350, cost_usd=0.001)

    # Retrieval
    retr1 = g.begin_node("Retrieval", input_payload={"q": "recent deploys checkout-api"}, parent_id=root)
    g.complete_node(retr1, output_payload={"deploys": ["v3.5.0", "v3.4.1"]},
                    latency_ms=14.5, token_count=0, cost_usd=0.0)

    # Tool 1: CloudWatch — succeeds
    tool1 = g.begin_node("Tool_Invocation",
                         input_payload={"tool": "cloudwatch.GetMetricStatistics"}, parent_id=retr1)
    import datetime as dt
    now = dt.datetime.now(dt.timezone.utc)
    response = cw.get_metric_statistics(
        Namespace="AWS/ApplicationELB", MetricName="TargetResponseTime",
        StartTime=now - dt.timedelta(hours=1), EndTime=now, Period=300, Statistics=["Maximum"],
    )
    g.complete_node(tool1, output_payload={"p99_ms": 8200, "datapoints_count": len(response.get("Datapoints", []))},
                    latency_ms=145.0, token_count=0, cost_usd=0.0)

    # Tool 2: EC2 describe — FAILS (invalid InstanceId format)
    tool2 = g.begin_node("Tool_Invocation",
                         input_payload={"tool": "ec2.DescribeInstances", "instance_id": "i-INVALID"},
                         parent_id=retr1)
    try:
        ec2.describe_instances(InstanceIds=["i-INVALID"])
        g.complete_node(tool2, output_payload={}, latency_ms=50.0)
    except ClientError as e:
        g.fail_node(tool2, error=f"ClientError: {e.response['Error']['Code']}")

    # LLM call
    llm1 = g.begin_node("LLM_Call",
                        input_payload={"prompt": "p99=8200ms, EC2 describe failed — hypothesize"},
                        parent_id=root)
    g.complete_node(llm1, output_payload={"hypothesis": "payments backpressure post-v3.5.0"},
                    latency_ms=2340.0, token_count=1200, cost_usd=0.018)

print(json.dumps(g.summary(), indent=2))

## Locate the failed node and reconstruct its causal chain

In [ ]:
failed_nodes = g.failed()
print(f"failed nodes: {len(failed_nodes)}")
assert len(failed_nodes) == 1, f"expected 1 failed node, got {len(failed_nodes)}"
fn = failed_nodes[0]
print(f"  - {fn.type} | error: {fn.error}")

chain = g.causal_chain(fn.id)
print(f"\ncausal chain from root ({len(chain)} nodes):")
for c in chain:
    print(f"  - {c.type} | id={c.id[:8]}")
assert chain[0].parent_id is None, "chain root must have no parent"
assert chain[-1].id == fn.id, "chain must end at failed node"
print("\n[PASS] Failed node located AND causal chain reconstructed from root.")

## Cypher-like query: find slow LLM calls

In [ ]:
slow_llms = g.query(lambda n: n.type == "LLM_Call"
                              and n.latency_ms is not None
                              and n.latency_ms > 1000)
print(f"slow LLM calls (> 1000ms): {len(slow_llms)}")
for n in slow_llms:
    print(f"  - latency={n.latency_ms:.1f}ms tokens={n.token_count} cost=${n.cost_usd}")
assert len(slow_llms) == 1, f"expected 1 slow LLM call, got {len(slow_llms)}"
print("\n[PASS] Cypher-like predicate query returned the slow LLM call.")

## Cost + token totals — visible at the execution-graph level

In [ ]:
s = g.summary()
print(json.dumps(s, indent=2))
assert s["failed_node_count"] == 1, "summary should report 1 failed node"
assert s["total_cost_usd"] > 0, "total cost should be > 0 after an LLM call"
assert s["total_token_count"] > 0, "total tokens should be > 0"
print("\n[PASS] Cost + token totals roll up correctly across the execution.")

## Conclusion
- Traced a multi-step DevOps investigation against `moto`-mocked AWS — CloudWatch metrics call succeeded, EC2 describe call failed (invalid InstanceId).
- The execution graph captured the failed node AND its causal chain back to root — the same diagnostic surface every Ch7 evaluation layer (0/1/2/3) will query.
- Cypher-like predicate queries work on the in-memory graph; production swaps in Neo4j without changing the contract.
- Cost + token totals roll up across the execution — feeds the Ch7 Reasoning Shape Analysis section.

**Without the execution graph, every Ch7 diagnostic technique is guesswork. With it, the agent's autobiography becomes a queryable substrate for self-evolution.**